In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 22. DPO와 최신 정렬 기법 — RLHF 없이 정렬하기

> **학습 목표**
> - DPO가 RLHF의 3단계를 1단계로 줄이는 수학을 유도한다
> - Bradley-Terry 모델과 DPO 손실의 관계를 이해한다
> - 최신 정렬 기법(KTO, IPO, ORPO)의 차이를 안다

## 22.1 RLHF의 복잡성과 DPO의 등장

RLHF의 문제:
- 4개 모델 (policy, reference, RM, value) → 메모리 폭발
- PPO 하이퍼파라미터 민감, 학습 불안정
- RM 따로 학습 → 에러 누적

**DPO** (Direct Preference Optimization, Rafailov et al., 2023):
- RM 학습 + PPO를 **단일 손실**로 통합
- 선호 데이터만으로 직접 학습
- 메모리: 2개 모델만 (policy, reference)


In [ ]:

# 공통 모델 정의 (Ch 18과 동일)
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model 정의 완료")


In [ ]:
# 어휘 크기 설정 (Ch 22 전체)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

vocab_size = 100  # toy
print("vocab_size (toy):", vocab_size)


## 22.2 Bradley-Terry 보상 모델 복습

$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

RM 손실: $\mathcal{L}_{\text{RM}} = -\log \sigma(r_w - r_l)$

## 22.3 보상-정책 관계식 유도

KL 제약 최적화 문제:
$$\max_\pi \mathbb{E}_{\mathbf{y} \sim \pi}[r(\mathbf{x}, \mathbf{y})] - \beta D_{\text{KL}}(\pi(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

이 최적화의 해 (라그랑지안 풀이):
$$\pi^*(\mathbf{y}|\mathbf{x}) = \frac{\pi_{\text{ref}}(\mathbf{y}|\mathbf{x}) \exp(r(\mathbf{x}, \mathbf{y}) / \beta)}{Z(\mathbf{x})}$$

정리하면:
$$r(\mathbf{x}, \mathbf{y}) = \beta \log \frac{\pi^*(\mathbf{y}|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}|\mathbf{x})} + \beta \log Z(\mathbf{x})$$

$\log Z(\mathbf{x})$는 $\mathbf{y}$와 무관하므로, $\mathbf{y}_w$ vs $\mathbf{y}_l$ 비교에서 상쇄.

## 22.4 DPO 손실 유도

보상-정책 관계를 Bradley-Terry에 대입:

$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma\left(\beta \log \frac{\pi_\theta(\mathbf{y}_w|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_w|\mathbf{x})} - \beta \log \frac{\pi_\theta(\mathbf{y}_l|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_l|\mathbf{x})}\right)$$

**DPO 손실**:
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(\mathbf{y}_w|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_w|\mathbf{x})} - \beta \log \frac{\pi_\theta(\mathbf{y}_l|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_l|\mathbf{x})}\right)\right]$$

핵심: **RM 없이** 정책 $\pi_\theta$를 직접 학습. 로그 확률 비율만 사용.


In [ ]:
# DPO 손실 구현
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def get_seq_logprob(model, x, y_mask=None):
    """시퀀스의 로그 Probability Sum."""
    logits = model(x)
    log_probs = F.log_softmax(logits, dim=-1)
    # shift
    log_probs = log_probs[:, :-1, :]
    targets = x[:, 1:]
    # gather
    seq_log_probs = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)  # (B, T-1)
    if y_mask is not None:
        seq_log_probs = seq_log_probs * y_mask[:, 1:]
    return seq_log_probs.sum(dim=-1)  # (B,)

def dpo_loss(policy_chosen_logp, policy_rejected_logp,
             ref_chosen_logp, ref_rejected_logp, beta=0.1):
    """DPO Loss."""
    chosen_ratio = policy_chosen_logp - ref_chosen_logp
    rejected_ratio = policy_rejected_logp - ref_rejected_logp
    logits = beta * (chosen_ratio - rejected_ratio)
    return -F.logsigmoid(logits).mean()

# 시뮬레이션
# 가짜 로그 Probability (실제로는 Model로 Computation)
torch.manual_seed(0)
B = 8
ref_chosen_logp = torch.randn(B) - 5  # 참조 Model의 chosen 로그 Probability
ref_rejected_logp = torch.randn(B) - 7  # rejected는 보통 더 낮음

# policy 초기: ref와 비슷
policy_chosen_logp = ref_chosen_logp.clone() + torch.randn(B) * 0.1
policy_rejected_logp = ref_rejected_logp.clone() + torch.randn(B) * 0.1

# Training 시뮬레이션 (policy를 chosen 쪽으로)
losses = []
for step in range(100):
    # policy를 Point진적으로 chosen 쪽으로 (gradient 시뮬레이션)
    policy_chosen_logp = policy_chosen_logp + 0.05
    # gradient ascent on chosen ratio

    loss = dpo_loss(policy_chosen_logp, policy_rejected_logp,
                    ref_chosen_logp, ref_rejected_logp, beta=0.1)
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('DPO Loss')
plt.title('DPO Training 시뮬레이션')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch22_dpo_loss.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"초기 loss: {losses[0]:.4f}, 최종 loss: {losses[-1]:.4f}")
print(f"chosen/rejected log-prob Difference: {(policy_chosen_logp - policy_rejected_logp).mean():.4f}")


## 22.5 DPO 학습 루프

핵심 단계:
1. SFT 모델을 $\pi_{\text{ref}}$로 고정
2. 선호 데이터 $(\mathbf{x}, \mathbf{y}_w, \mathbf{y}_l)$ 배치
3. $\pi_\theta$와 $\pi_{\text{ref}}$ 각각에서 $\mathbf{y}_w, \mathbf{y}_l$의 로그 확률 계산
4. DPO 손실 계산, $\pi_\theta$만 역전파


In [ ]:
# DPO 학습 (간소화)
import torch
import torch.nn as nn
import torch.nn.functional as F

# 가짜 선호 데이터 생성
def make_pref_batch(vocab_size, batch_size=4, seq_len=32):
    """chosen과 rejected 시퀀스 Generation."""
    chosen = torch.randint(0, vocab_size, (batch_size, seq_len))
    # rejected는 chosen에서 일부 토큰 바꿈
    rejected = chosen.clone()
    n_change = seq_len // 4
    for i in range(batch_size):
        idxs = torch.randperm(seq_len)[:n_change]
        rejected[i, idxs] = torch.randint(0, vocab_size, (n_change,))
    return chosen, rejected

# Model 2개 (policy, reference)
torch.manual_seed(0)
policy = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=64)
ref = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=64)
ref.load_state_dict(policy.state_dict())  # 초기에는 같음
for p in ref.parameters():
    p.requires_grad = False  # ref는 고정

opt = torch.optim.AdamW(policy.parameters(), lr=5e-4, weight_decay=0.01)

losses = []
for step in range(50):
    chosen, rejected = make_pref_batch(vocab_size, batch_size=4, seq_len=32)
    # 로그 Probability (시퀀스 전체)
    with torch.no_grad():
        ref_chosen_logp = get_seq_logprob(ref, chosen)
        ref_rejected_logp = get_seq_logprob(ref, rejected)
    policy_chosen_logp = get_seq_logprob(policy, chosen)
    policy_rejected_logp = get_seq_logprob(policy, rejected)

    loss = dpo_loss(policy_chosen_logp, policy_rejected_logp,
                    ref_chosen_logp, ref_rejected_logp, beta=0.1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('DPO Loss')
plt.title('DPO Training (MiniGPT)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch22_dpo_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"최종 DPO loss: {losses[-1]:.4f}")


## 22.6 IPO, KTO, ORPO — DPO의 변형

### IPO (Identity Preference Optimization)
DPO가 과적합되는 문제 해결:
$$\mathcal{L}_{\text{IPO}} = (\log \sigma(\ldots) - \frac{1}{2})^2$$

### KTO (Kahneman-Tversky Optimization)
**선호 쌍이 필요 없음**. 좋은/나쁜 응답 각각 라벨만 있으면 됨:
$$\mathcal{L}_{\text{KTO}} = \sigma(\beta \log \frac{\pi}{\pi_{\text{ref}}} - z) + \lambda \sigma(z - \beta \log \frac{\pi}{\pi_{\text{ref}}})$$

데이터 구하기 훨씬 쉬움 (쌍 매칭 불필요).

### ORPO (Odds Ratio Preference Optimization)
SFT와 DPO를 통합 — reference model조차 필요 없음. 가장 효율적.


## 22.7 DPO vs PPO 비교

| 항목 | PPO (RLHF) | DPO |
|---|---|---|
| 모델 수 | 4 (policy, ref, RM, value) | 2 (policy, ref) |
| 학습 안정성 | 불안정 | 안정적 |
| 하이퍼파라미터 | 많음 | 적음 (β 정도) |
| 메모리 | 큼 | 작음 |
| 구현 복잡도 | 높음 | 낮음 |
| 품질 | 약간 더 좋을 수 있음 | 비슷 |

실무에서는 DPO가 점점 표준. 최신 모델(Mistral, LLaMA-3 등)도 DPO 사용.


## 22.8 핵심 정리

| 기법 | 핵심 아이디어 | 데이터 |
|---|---|---|
| RLHF | RM + PPO + KL | 선호 쌍 |
| DPO | 보상-정책 관계식으로 단일 손실 | 선호 쌍 |
| IPO | DPO + 정규화 | 선호 쌍 |
| KTO | 좋은/나쁜 라벨만 | 단일 라벨 |
| ORPO | SFT+DPO 통합 | 선호 쌍 |

## 연습문제

1. DPO 손실에서 $\beta = 0.01, 0.1, 1.0$을 비교하고 효과를 설명하라.
2. Bradley-Terry 모델이 "선호 쌍"을 어떻게 모델링하는지 설명하라.
3. DPO와 PPO의 메모리 사용량을 비교하라.
4. KTO가 왜 선호 쌍 없이 학습 가능한지 설명하라.
5. 보상-정책 관계식 $r = \beta \log \frac{\pi^*}{\pi_{\text{ref}}} + \beta \log Z$를 유도하라.

> 해설: `solutions/ch22_solutions.ipynb`
